# Stage 9C — Prepare a reproducible daily training sample

**Goal:** Read the complete 485-day raw dataset, choose the same 10% of request IDs every time, make 72-hour labels, and save one prepared file per day.

**Why:** The raw collection has 4,982,820 requests. A fixed 10% sample keeps the next training exercise manageable in free Colab while retaining coverage from every calendar day. We do not select days or outcomes based on what performed well.

We keep all 4,982,820 raw rows. The sample is only for this model experiment. The March–April 2026 test labels remain sealed during model selection. Cell 6 has a 10-minute time budget and can be rerun.

In [ ]:
# Cell 2 — Why: select and verify the complete raw dataset manifest.
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, time, pandas as pd

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
dataset_id = "20260926T141310793030Z"
dataset_dir = root / "datasets" / dataset_id
dataset = json.loads((dataset_dir / "manifest.json").read_text())
if dataset["dataset_id"] != dataset_id or len(dataset["partitions"]) != 485:
    raise ValueError("Wrong or incomplete raw dataset")
if sum(part["rows"] for part in dataset["partitions"]) != dataset["total_rows"]:
    raise ValueError("Raw dataset row count does not match its manifest")
as_of = pd.Timestamp(dataset["created_at_utc"])
prepared_root = root / "prepared_daily" / dataset_id
prepared_root.mkdir(parents=True, exist_ok=True)
print("Raw days:", len(dataset["partitions"]), "Rows:", dataset["total_rows"])
print("Snapshot as of:", as_of)

In [ ]:
# Cell 3 — Why: select IDs by a fixed rule, independent of their labels.
def selected(request_id):
    digest = hashlib.sha256(str(request_id).encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "big") % 10 == 0

print("Sampling rule: first 64 SHA-256 bits of unique_key, modulo 10 equals 0")
print("Approximate selection rate: 10% on every day")

In [ ]:
# Cell 4 — Why: apply the same 72-hour label rule to each day.
def label_sample(rows):
    columns = ["unique_key", "created_date", "closed_date", "complaint_type", "borough"]
    frame = pd.DataFrame(rows).reindex(columns=columns)
    created_local = pd.to_datetime(frame["created_date"], errors="coerce")
    closed_local = pd.to_datetime(frame["closed_date"], errors="coerce")
    created = created_local.dt.tz_localize(
        "America/New_York", ambiguous="NaT", nonexistent="NaT"
    ).dt.tz_convert("UTC")
    closed = closed_local.dt.tz_localize(
        "America/New_York", ambiguous="NaT", nonexistent="NaT"
    ).dt.tz_convert("UTC")
    deadline = created + pd.Timedelta(hours=72)
    bad_time = created.isna() | (frame["closed_date"].notna() & closed.isna())
    bad_time |= closed.notna() & ((closed < created) | (closed > as_of))
    unknown = ~bad_time & (deadline > as_of)
    missing_input = frame["complaint_type"].isna() | frame["borough"].isna()
    missing_input |= frame["complaint_type"].fillna("").astype(str).str.strip().eq("")
    missing_input |= frame["borough"].fillna("").astype(str).str.strip().eq("")
    keep = ~(bad_time | unknown | missing_input)
    result = frame.loc[keep, ["unique_key", "created_date", "complaint_type", "borough"]].copy()
    result["month"] = created_local.loc[keep].dt.month.to_numpy()
    result["hour"] = created_local.loc[keep].dt.hour.to_numpy()
    result["weekday"] = created_local.loc[keep].dt.weekday.to_numpy()
    result["label"] = (closed.isna() | (closed > deadline)).loc[keep].astype(int).to_numpy()
    counts = {"sampled": len(frame), "saved": len(result),
              "invalid_time": int(bad_time.sum()), "outcome_unknown": int(unknown.sum()),
              "missing_input": int(missing_input.sum())}
    return result, counts

print("Label function ready; closed_date is used only to form label")

In [ ]:
# Cell 5 — Why: verify raw bytes and save a prepared day with lineage.
def prepare_day(part):
    day = part["day"]
    folder = prepared_root / day
    output_file, meta_file = folder / "rows.jsonl", folder / "manifest.json"
    if meta_file.exists():
        meta = json.loads(meta_file.read_text())
        if meta["day"] != day or meta["source_sha256"] != part["sha256"]:
            raise ValueError(f"{day}: prepared lineage changed")
        return meta, "existing"

    raw = (root / part["file"]).read_bytes()
    if hashlib.sha256(raw).hexdigest() != part["sha256"] or len(raw.splitlines()) != part["rows"]:
        raise ValueError(f"{day}: raw file changed")
    chosen = []
    for line in raw.splitlines():
        row = json.loads(line)
        if selected(row["unique_key"]):
            chosen.append(row)
    labeled, counts = label_sample(chosen)
    clean = labeled.astype(object).where(pd.notna(labeled), None)
    content = "".join(json.dumps(row, sort_keys=True, allow_nan=False) + "\n"
                      for row in clean.to_dict(orient="records")).encode("utf-8")
    meta = {"day": day, "source_dataset_id": dataset_id,
            "source_sha256": part["sha256"], "sha256": hashlib.sha256(content).hexdigest(),
            "counts": counts, "sample_rule": "sha256(unique_key) first 64 bits modulo 10 == 0",
            "label_rule": "1 if not closed within 72 actual hours; else 0",
            "time_zone_assumption": "America/New_York", "lesson_version": "stage9c-v1"}
    folder.mkdir(parents=True, exist_ok=True)
    pending = folder / "rows.jsonl.pending"
    pending.write_bytes(content)
    if hashlib.sha256(pending.read_bytes()).hexdigest() != meta["sha256"]:
        raise ValueError(f"{day}: prepared write failed")
    pending.replace(output_file)
    meta_pending = folder / "manifest.json.pending"
    meta_pending.write_text(json.dumps(meta, indent=2) + "\n", encoding="utf-8")
    meta_pending.replace(meta_file)
    return meta, "saved"

print("Prepared-day saver ready")

In [ ]:
# Cell 6 — Why: work in bounded runs and resume saved days.
deadline = time.monotonic() + 10 * 60
new_days = 0
for part in dataset["partitions"]:
    if time.monotonic() >= deadline:
        print("Time budget reached. Rerun Cell 6 to continue.")
        break
    if (prepared_root / part["day"] / "manifest.json").exists():
        continue
    meta, action = prepare_day(part)
    new_days += 1
    if new_days % 25 == 0:
        print("Prepared new days:", new_days, "Latest:", part["day"], flush=True)
print("New prepared days this run:", new_days)

In [ ]:
# Cell 7 — Why: show progress without exposing test outcomes.
days = [part["day"] for part in dataset["partitions"]]
ready = [day for day in days if (prepared_root / day / "manifest.json").exists()]
missing = [day for day in days if day not in set(ready)]
print("Prepared days:", len(ready), "of", len(days))
print("Remaining days:", len(missing))
print("Next missing day:", missing[0] if missing else "none")
print("No test label rates displayed")

In [ ]:
# Cell 8 — Why: make a prepared dataset ID after all daily checks pass.
if missing:
    print("Prepared dataset unfinished; rerun Cell 6 later.")
else:
    prepared_parts = []
    totals = {"sampled": 0, "saved": 0, "invalid_time": 0,
              "outcome_unknown": 0, "missing_input": 0}
    for number, part in enumerate(dataset["partitions"], 1):
        folder = prepared_root / part["day"]
        meta = json.loads((folder / "manifest.json").read_text())
        file = folder / "rows.jsonl"
        raw = file.read_bytes()
        if hashlib.sha256((root / part["file"]).read_bytes()).hexdigest() != part["sha256"]:
            raise ValueError(f"{part['day']}: raw source changed")
        if meta["source_sha256"] != part["sha256"] or hashlib.sha256(raw).hexdigest() != meta["sha256"]:
            raise ValueError(f"{part['day']}: final prepared verification failed")
        if len(raw.splitlines()) != meta["counts"]["saved"]:
            raise ValueError(f"{part['day']}: prepared row count changed")
        prepared_parts.append({"day": part["day"], "file": str(file.relative_to(root)),
                               "sha256": meta["sha256"], "rows": meta["counts"]["saved"]})
        for key in totals:
            totals[key] += meta["counts"][key]
        if number % 50 == 0 or number == len(days):
            print("Verified prepared files:", number, "of", len(days), flush=True)
    prepared_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    prepared_dir = root / "prepared_datasets" / prepared_id
    prepared_dir.mkdir(parents=True, exist_ok=False)
    prepared_manifest = {"prepared_id": prepared_id, "source_dataset_id": dataset_id,
                         "as_of_utc": str(as_of), "parts": prepared_parts,
                         "totals": totals, "sample_rule": "sha256(unique_key) first 64 bits modulo 10 == 0",
                         "test_sealed_for_selection": True, "lesson_version": "stage9c-v1"}
    (prepared_dir / "manifest.json").write_text(json.dumps(prepared_manifest, indent=2) + "\n", encoding="utf-8")
    print("Prepared dataset ID:", prepared_id)
    print("Sampled/saved/invalid/unknown/missing:", totals)

# Stage 9C takeaway and quiz

**What you learned:** We kept full raw coverage, selected a fixed sample by request ID, applied the same 72-hour rule, excluded invalid or not-yet-known outcomes, and saved provenance for every prepared day. Sampling by ID does not choose cases based on their answer.

**Easy quiz:**

1. Why hash `unique_key` instead of randomly sampling again each run?
2. Did we throw away the other 90% of raw requests?
3. Why is `closed_date` used to make the label but absent from saved model inputs?
4. Why do we avoid printing test label rates now?

Send me **Prepared days**, **Remaining days**, and, once complete, **Prepared dataset ID** and the totals. Next we will create the new chronological split and train a candidate.